In [1]:
import inspect
import transformers

from transformers.models.wav2vec2_conformer.modeling_wav2vec2_conformer import (
    Wav2Vec2ConformerConvolutionModule,
    Wav2Vec2ConformerEncoderLayer
)

print("Transformers version:", transformers.__version__)

# 1. 查看 ConvolutionModule 源码位置
cls = Wav2Vec2ConformerConvolutionModule

print("\n===== ConvolutionModule =====")
print("Source file:", inspect.getfile(cls))

source_lines, start_line = inspect.getsourcelines(cls)

print(f"Line range: {start_line}-{start_line + len(source_lines) - 1}")

print("\n===== Source Code =====")
print("".join(source_lines))

Transformers version: 5.21.0

===== ConvolutionModule =====
Source file: d:\anaconda3\envs\speech-ai\Lib\site-packages\transformers\models\wav2vec2_conformer\modeling_wav2vec2_conformer.py
Line range: 366-423

===== Source Code =====
class Wav2Vec2ConformerConvolutionModule(nn.Module):
    """Convolution block used in the conformer block"""

    def __init__(self, config):
        super().__init__()
        if (config.conv_depthwise_kernel_size - 1) % 2 == 1:
            raise ValueError("`config.conv_depthwise_kernel_size` should be a odd number for 'SAME' padding")
        self.layer_norm = nn.LayerNorm(config.hidden_size)
        self.pointwise_conv1 = nn.Conv1d(
            config.hidden_size,
            2 * config.hidden_size,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )
        self.glu = nn.GLU(dim=1)
        self.depthwise_conv = nn.Conv1d(
            config.hidden_size,
            config.hidden_size,
            c

In [2]:
# 2. 查看完整 Encoder Layer 的 forward
print(inspect.getsource(Wav2Vec2ConformerEncoderLayer.forward))

    def forward(
        self,
        hidden_states,
        attention_mask: torch.Tensor | None = None,
        relative_position_embeddings: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ):
        # 1. Feed-Forward 1 layer
        residual = hidden_states
        hidden_states = self.ffn1_layer_norm(hidden_states)
        hidden_states = self.ffn1(hidden_states)
        hidden_states = hidden_states * 0.5 + residual
        residual = hidden_states

        # 2. Self-Attention layer
        hidden_states = self.self_attn_layer_norm(hidden_states)
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            relative_position_embeddings=relative_position_embeddings,
            **kwargs,
        )
        hidden_states = self.self_attn_dropout(hidden_states)
        hidden_states = hidden_states + residual

        # 3. Convolutional Layer
        residual = hidden_states
   